# 🌐 Multilingual Translation: mmBERT-small + mT5-small

**6 chiều dịch:** `en↔vi` | `en↔fr` | `vi↔fr`

**Cơ chế:** Language Token `>>vi<<` `>>en<<` `>>fr<<` ghép vào đầu câu nguồn

```
>>vi<< Hello world        →  Xin chào thế giới
>>en<< Bonjour            →  Hello
>>fr<< Xin chào thế giới  →  Bonjour tout le monde
```

## ⚙️ Cell 1 – Cài đặt

In [ ]:
!pip install transformers datasets sentencepiece sacrebleu accelerate -q
# !pip install flash-attn --no-build-isolation -q  # Tuỳ chọn
print("✅ Xong")

## 💾 Cell 2 – Mount Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
import os
OUTPUT_DIR = "/content/drive/MyDrive/translation_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"✅ Drive OK | {OUTPUT_DIR}")

## 📝 Cell 3 – Ghi model.py

In [ ]:
%%writefile /content/model.py
"""
model.py – Kiến trúc dịch máy đa ngôn ngữ
Encoder : jhu-clsp/mmBERT-small  (ModernBERT-based, hidden=384, Gemma-2 tokenizer)
Decoder : google/mt5-small        (d_model=512)
Bridge  : EncoderProjection 384 → 512

Hỗ trợ 6 chiều dịch bằng language token:
    >>vi<<  >>en<<  >>fr<<  ghép vào đầu câu nguồn
    Ví dụ: ">>vi<< Hello world" → model biết cần dịch sang tiếng Việt
"""

import torch
import torch.nn as nn
from transformers import AutoModel, MT5ForConditionalGeneration
from transformers.modeling_outputs import BaseModelOutput


# ──────────────────────────────────────────────────────────────
# 1. ENCODER – mmBERT-small
# ──────────────────────────────────────────────────────────────
class mmBERTEncoder(nn.Module):
    MMBERT_SMALL = "jhu-clsp/mmBERT-small"
    MMBERT_BASE  = "jhu-clsp/mmBERT-base"

    def __init__(self, model_name: str = MMBERT_SMALL, freeze: bool = False):
        super().__init__()
        try:
            self.bert = AutoModel.from_pretrained(
                model_name,
                attn_implementation="flash_attention_2",
                torch_dtype=torch.bfloat16,
            )
            print("  ✅ Encoder: Flash Attention 2 được kích hoạt")
        except Exception:
            self.bert = AutoModel.from_pretrained(model_name)
            print("  ℹ️  Encoder: dùng Eager Attention")

        self.hidden_size = self.bert.config.hidden_size  # 384

        if freeze:
            for p in self.bert.parameters():
                p.requires_grad = False
            print("  🔒 Encoder bị đóng băng")

    def forward(self, input_ids, attention_mask):
        out = self.bert(input_ids=input_ids, attention_mask=attention_mask)
        return out.last_hidden_state  # (B, L, 384)


# ──────────────────────────────────────────────────────────────
# 2. PROJECTION – 384 → 512
# ──────────────────────────────────────────────────────────────
class EncoderProjection(nn.Module):
    def __init__(self, encoder_dim: int, decoder_dim: int, dropout: float = 0.1):
        super().__init__()
        linear = nn.Linear(encoder_dim, decoder_dim)
        nn.init.xavier_uniform_(linear.weight)
        nn.init.zeros_(linear.bias)
        self.proj = nn.Sequential(
            linear,
            nn.LayerNorm(decoder_dim),
            nn.Dropout(dropout),
        )

    def forward(self, x):
        return self.proj(x)  # (B, L, 512)


# ──────────────────────────────────────────────────────────────
# 3. MÔ HÌNH ĐẦY ĐỦ
# ──────────────────────────────────────────────────────────────
class TranslationModel(nn.Module):
    """
    Một model duy nhất hỗ trợ tất cả cặp ngôn ngữ.
    Chiều dịch được điều khiển bởi language token trong câu nguồn:

        ">>vi<< Hello world"  →  "Xin chào thế giới"
        ">>en<< Bonjour"      →  "Hello"
        ">>fr<< Xin chào"     →  "Bonjour"

    Các cặp hỗ trợ: en↔vi, en↔fr, vi↔fr  (6 chiều)
    """

    # Language tokens (ghép vào đầu câu nguồn)
    LANG_TOKENS = {
        "en": ">>en<<",
        "vi": ">>vi<<",
        "fr": ">>fr<<",
    }

    # 6 chiều dịch được hỗ trợ
    SUPPORTED_PAIRS = [
        ("en", "vi"), ("vi", "en"),
        ("en", "fr"), ("fr", "en"),
        ("vi", "fr"), ("fr", "vi"),
    ]

    DEFAULT_ENCODER = "jhu-clsp/mmBERT-small"
    DEFAULT_DECODER = "google/mt5-small"

    def __init__(
        self,
        encoder_name: str = DEFAULT_ENCODER,
        decoder_name: str = DEFAULT_DECODER,
        freeze_encoder: bool = False,
        proj_dropout: float = 0.1,
    ):
        super().__init__()
        print(f"\nKhởi tạo TranslationModel (đa ngôn ngữ):")
        print(f"  Encoder : {encoder_name}")
        print(f"  Decoder : {decoder_name}")
        print(f"  Hỗ trợ  : {len(self.SUPPORTED_PAIRS)} chiều dịch")

        self.encoder    = mmBERTEncoder(encoder_name, freeze=freeze_encoder)
        encoder_dim     = self.encoder.hidden_size        # 384

        self.mt5        = MT5ForConditionalGeneration.from_pretrained(decoder_name)
        decoder_dim     = self.mt5.config.d_model         # 512

        self.projection = EncoderProjection(encoder_dim, decoder_dim, proj_dropout)
        print(f"  Projection: {encoder_dim} → {decoder_dim}")

    @staticmethod
    def add_lang_token(text: str, tgt_lang: str) -> str:
        """Ghép language token vào đầu câu nguồn."""
        token = TranslationModel.LANG_TOKENS.get(tgt_lang, f">>{tgt_lang}<<")
        return f"{token} {text}"

    def forward(self, input_ids, attention_mask, labels=None,
                decoder_input_ids=None):
        enc_hidden      = self.encoder(input_ids, attention_mask)
        enc_projected   = self.projection(enc_hidden)
        encoder_outputs = BaseModelOutput(last_hidden_state=enc_projected)
        return self.mt5(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            decoder_input_ids=decoder_input_ids,
            labels=labels,
        )

    @torch.no_grad()
    def translate(self, input_ids, attention_mask,
                  max_new_tokens=128, num_beams=4,
                  length_penalty=1.0, early_stopping=True, **kwargs):
        enc_hidden      = self.encoder(input_ids, attention_mask)
        enc_projected   = self.projection(enc_hidden)
        encoder_outputs = BaseModelOutput(last_hidden_state=enc_projected)
        return self.mt5.generate(
            encoder_outputs=encoder_outputs,
            attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams,
            length_penalty=length_penalty,
            early_stopping=early_stopping,
            **kwargs,
        )

    def count_parameters(self):
        fmt = lambda n: f"{n/1e6:.2f}M"
        enc  = sum(p.numel() for p in self.encoder.parameters())
        proj = sum(p.numel() for p in self.projection.parameters())
        dec  = sum(p.numel() for p in self.mt5.parameters())
        trn  = sum(p.numel() for p in self.parameters() if p.requires_grad)
        return {
            "mmBERT-small encoder" : fmt(enc),
            "projection (384→512)" : fmt(proj),
            "mT5-small decoder"    : fmt(dec),
            "total"                : fmt(enc + proj + dec),
            "trainable"            : fmt(trn),
        }


# ──────────────────────────────────────────────────────────────
# 4. SANITY CHECK
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    model = TranslationModel().to(device)
    for k, v in model.count_parameters().items():
        print(f"  {k:30s}: {v}")

    # Kiểm tra language token
    for src, tgt in TranslationModel.SUPPORTED_PAIRS:
        example = TranslationModel.add_lang_token("Hello world", tgt)
        print(f"  {src}→{tgt}: '{example}'")


## 📝 Cell 4 – Ghi train.py

In [ ]:
%%writefile /content/train.py
"""
train.py – Training đa ngôn ngữ (6 chiều dịch)
Hỗ trợ: en↔vi, en↔fr, vi↔fr

Cơ chế: Language Token ghép vào đầu câu nguồn
    ">>vi<< Hello"  →  model biết dịch sang tiếng Việt
    ">>en<< Bonjour" →  model biết dịch sang tiếng Anh

Dữ liệu: opus-100
    en-vi : ~1M cặp  → lấy train_samples_per_pair
    en-fr : ~1M cặp
    fr-vi : ít hơn   → zero-shot hoặc pivot qua en

Cài đặt:
    !pip install transformers datasets sentencepiece sacrebleu accelerate -q
"""

import os, math, time, json, random
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader, ConcatDataset
from torch.optim import AdamW
from transformers import AutoTokenizer, get_linear_schedule_with_warmup
from datasets import load_dataset
import sacrebleu

from model import TranslationModel

# ──────────────────────────────────────────────────────────────
# 1. HYPERPARAMETERS
# ──────────────────────────────────────────────────────────────
CFG = {
    "encoder_name":   "jhu-clsp/mmBERT-small",
    "decoder_name":   "google/mt5-small",
    "freeze_encoder": False,

    # ── Các cặp ngôn ngữ cần train ──────────────────────────
    # Mỗi cặp (src, tgt) sẽ tự động tạo cả 2 chiều A→B và B→A
    # opus-100 có sẵn: en-vi, en-fr
    # fr-vi không có trực tiếp → dùng pivot (en làm cầu nối)
    "lang_pairs": [
        ("en", "vi"),   # → sinh en→vi và vi→en
        ("en", "fr"),   # → sinh en→fr và fr→en
        # ("fr", "vi"), # uncomment nếu muốn thêm fr↔vi trực tiếp
    ],

    # Số mẫu mỗi chiều dịch (None = toàn bộ)
    "train_samples_per_pair": 30_000,   # × 4 chiều = 120k total
    "val_samples_per_pair":   500,

    "max_src_len":    128,
    "max_tgt_len":    128,

    "batch_size":     8,
    "grad_accum":     16,       # Effective batch = 128
    "epochs":         5,
    "lr":             5e-5,
    "warmup_ratio":   0.1,
    "weight_decay":   0.01,
    "max_grad_norm":  1.0,

    "save_every_minutes": 10,
    "output_dir": "/content/drive/MyDrive/translation_model",
    "log_file":   "/content/drive/MyDrive/translation_model/train_log.json",
}

torch.manual_seed(42)
random.seed(42)


# ──────────────────────────────────────────────────────────────
# 2. DATASET
# ──────────────────────────────────────────────────────────────
class TranslationDataset(Dataset):
    """
    Mỗi sample: câu nguồn có language token + câu đích.
    Ví dụ:
        input : ">>vi<< Hello world"      (en→vi)
        label : "Xin chào thế giới"
    """

    def __init__(self, pairs, enc_tokenizer, dec_tokenizer,
                 max_src_len=128, max_tgt_len=128):
        self.pairs       = pairs          # [{"src": ..., "tgt": ...}, ...]
        self.enc_tok     = enc_tokenizer
        self.dec_tok     = dec_tokenizer
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len

    def __len__(self):
        return len(self.pairs)

    def __getitem__(self, idx):
        # src đã chứa language token: ">>vi<< ..."
        src = self.pairs[idx]["src"]
        tgt = self.pairs[idx]["tgt"]

        enc = self.enc_tok(src, max_length=self.max_src_len,
                           padding="max_length", truncation=True,
                           return_tensors="pt")
        dec = self.dec_tok(tgt, max_length=self.max_tgt_len,
                           padding="max_length", truncation=True,
                           return_tensors="pt")

        labels = dec["input_ids"].squeeze(0).clone()
        labels[labels == self.dec_tok.pad_token_id] = -100

        if (labels != -100).sum() == 0:
            labels[0] = dec["input_ids"].squeeze(0)[0]

        return {
            "input_ids":      enc["input_ids"].squeeze(0),
            "attention_mask": enc["attention_mask"].squeeze(0),
            "labels":         labels,
        }


def load_pairs_bidirectional(lang_a, lang_b, split, max_samples=None):
    """
    Tải opus-100 cho cặp (lang_a, lang_b) và tạo dữ liệu 2 chiều:
        A→B: src = ">>B<< câu_A",  tgt = câu_B
        B→A: src = ">>A<< câu_B",  tgt = câu_A
    """
    key = f"{lang_a}-{lang_b}"
    print(f"  Tải opus-100 [{key}] split={split}...")

    try:
        ds = load_dataset("Helsinki-NLP/opus-100", key, split=split)
    except Exception:
        try:
            ds = load_dataset("Helsinki-NLP/opus-100",
                              f"{lang_b}-{lang_a}", split=split)
        except Exception:
            print(f"  ⚠️  Không tìm thấy {key}, bỏ qua")
            return [], []

    if max_samples:
        ds = ds.select(range(min(max_samples, len(ds))))

    pairs_ab, pairs_ba = [], []
    lang_token_b = TranslationModel.LANG_TOKENS.get(lang_b, f">>{lang_b}<<")
    lang_token_a = TranslationModel.LANG_TOKENS.get(lang_a, f">>{lang_a}<<")

    for item in ds:
        tr   = item["translation"]
        text_a = tr.get(lang_a) or ""
        text_b = tr.get(lang_b) or ""
        if not text_a or not text_b:
            continue

        # Chiều A → B
        pairs_ab.append({
            "src": f"{lang_token_b} {text_a}",
            "tgt": text_b,
            "direction": f"{lang_a}→{lang_b}",
        })
        # Chiều B → A
        pairs_ba.append({
            "src": f"{lang_token_a} {text_b}",
            "tgt": text_a,
            "direction": f"{lang_b}→{lang_a}",
        })

    print(f"    → {lang_a}→{lang_b}: {len(pairs_ab):,} | "
          f"{lang_b}→{lang_a}: {len(pairs_ba):,}")
    return pairs_ab, pairs_ba


def build_all_pairs(lang_pairs, split, max_per_pair=None):
    """Tổng hợp tất cả cặp ngôn ngữ thành một list."""
    all_pairs = []
    stats = {}

    for lang_a, lang_b in lang_pairs:
        pairs_ab, pairs_ba = load_pairs_bidirectional(
            lang_a, lang_b, split, max_per_pair
        )
        all_pairs.extend(pairs_ab)
        all_pairs.extend(pairs_ba)
        stats[f"{lang_a}→{lang_b}"] = len(pairs_ab)
        stats[f"{lang_b}→{lang_a}"] = len(pairs_ba)

    # Shuffle để trộn đều các cặp
    random.shuffle(all_pairs)

    print(f"\n  Thống kê [{split}]:")
    for direction, count in stats.items():
        print(f"    {direction}: {count:,}")
    print(f"  Tổng: {len(all_pairs):,} cặp câu")
    return all_pairs


# ──────────────────────────────────────────────────────────────
# 3. METRICS
# ──────────────────────────────────────────────────────────────
def compute_bleu(predictions, references):
    return sacrebleu.corpus_bleu(predictions, [references]).score


# ──────────────────────────────────────────────────────────────
# 4. VALIDATION
# ──────────────────────────────────────────────────────────────
@torch.no_grad()
def validate(model, loader, dec_tokenizer, device, num_beams=4):
    model.eval()
    total_loss, steps = 0.0, 0
    preds, refs = [], []

    for batch in loader:
        input_ids      = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels         = batch["labels"].to(device)

        with torch.cuda.amp.autocast(dtype=torch.bfloat16,
                                     enabled=device.type == "cuda"):
            out = model(input_ids=input_ids,
                        attention_mask=attention_mask, labels=labels)

        total_loss += out.loss.item()
        steps += 1

        if len(preds) < 200:
            generated = model.translate(
                input_ids, attention_mask,
                max_new_tokens=64, num_beams=num_beams
            )
            decoded   = dec_tokenizer.batch_decode(generated,
                                                    skip_special_tokens=True)
            lbl_ids   = labels.clone()
            lbl_ids[lbl_ids == -100] = dec_tokenizer.pad_token_id
            ref_dec   = dec_tokenizer.batch_decode(lbl_ids,
                                                    skip_special_tokens=True)
            preds.extend(decoded)
            refs.extend(ref_dec)

    avg_loss = total_loss / steps if steps > 0 else 0.0
    bleu     = compute_bleu(preds, refs) if preds else 0.0
    model.train()
    return avg_loss, bleu


# ──────────────────────────────────────────────────────────────
# 5. SAVE CHECKPOINT
# ──────────────────────────────────────────────────────────────
def save_ckpt(path, epoch, step, epoch_loss_accum,
              model, optimizer, scheduler, best_bleu, log_history):
    torch.save({
        "epoch":            epoch,
        "step":             step,
        "epoch_loss_accum": epoch_loss_accum,
        "model":            model.state_dict(),
        "optimizer":        optimizer.state_dict(),
        "scheduler":        scheduler.state_dict(),
        "best_bleu":        best_bleu,
        "log_history":      log_history,
        "cfg":              CFG,
    }, path)


# ──────────────────────────────────────────────────────────────
# 6. TRAINING LOOP
# ──────────────────────────────────────────────────────────────
def train():
    os.makedirs(CFG["output_dir"], exist_ok=True)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")
    if device.type == "cuda":
        print(f"GPU  : {torch.cuda.get_device_name(0)}")
        print(f"VRAM : {torch.cuda.get_device_properties(0).total_memory/1e9:.1f} GB")

    # ── Tokenizers ─────────────────────────────────────────
    print("\nTải tokenizers...")
    enc_tokenizer = AutoTokenizer.from_pretrained(CFG["encoder_name"])
    dec_tokenizer = AutoTokenizer.from_pretrained(CFG["decoder_name"])
    print(f"  Encoder vocab: {enc_tokenizer.vocab_size:,}  (Gemma-2)")
    print(f"  Decoder vocab: {dec_tokenizer.vocab_size:,}  (SentencePiece)")

    # ── Data: tất cả cặp ngôn ngữ ──────────────────────────
    print("\nTải dữ liệu đa ngôn ngữ...")
    print(f"Cặp ngôn ngữ: {CFG['lang_pairs']}")

    train_pairs = build_all_pairs(
        CFG["lang_pairs"], "train", CFG["train_samples_per_pair"]
    )
    val_pairs = build_all_pairs(
        CFG["lang_pairs"], "validation", CFG["val_samples_per_pair"]
    )

    train_ds = TranslationDataset(train_pairs, enc_tokenizer, dec_tokenizer,
                                   CFG["max_src_len"], CFG["max_tgt_len"])
    val_ds   = TranslationDataset(val_pairs,   enc_tokenizer, dec_tokenizer,
                                   CFG["max_src_len"], CFG["max_tgt_len"])

    train_loader = DataLoader(
        train_ds, batch_size=CFG["batch_size"], shuffle=True,
        num_workers=2, pin_memory=True,
        generator=torch.Generator().manual_seed(42),
    )
    val_loader = DataLoader(
        val_ds, batch_size=CFG["batch_size"] * 2,
        shuffle=False, num_workers=2, pin_memory=True,
    )
    print(f"\nTrain: {len(train_ds):,} | Val: {len(val_ds):,}")

    # ── Model ──────────────────────────────────────────────
    model = TranslationModel(
        encoder_name=CFG["encoder_name"],
        decoder_name=CFG["decoder_name"],
        freeze_encoder=CFG["freeze_encoder"],
    ).to(device)

    print("\nThông số mô hình:")
    for k, v in model.count_parameters().items():
        print(f"  {k:30s}: {v}")

    # ── Optimizer & Scheduler ──────────────────────────────
    optimizer    = AdamW(
        [p for p in model.parameters() if p.requires_grad],
        lr=CFG["lr"], weight_decay=CFG["weight_decay"],
    )
    total_steps  = (len(train_loader) // CFG["grad_accum"]) * CFG["epochs"]
    warmup_steps = int(total_steps * CFG["warmup_ratio"])
    scheduler    = get_linear_schedule_with_warmup(
        optimizer, warmup_steps, total_steps
    )
    scaler = torch.cuda.amp.GradScaler(enabled=False)

    # ── Resume checkpoint ──────────────────────────────────
    ckpt_path        = os.path.join(CFG["output_dir"], "checkpoint_latest.pt")
    start_epoch      = 0
    start_step       = 0
    best_bleu        = 0.0
    log_history      = []
    epoch_loss_accum = 0.0

    if os.path.exists(ckpt_path):
        print(f"\nTìm thấy checkpoint: {ckpt_path}")
        ckpt = torch.load(ckpt_path, map_location=device)
        model.load_state_dict(ckpt["model"])
        optimizer.load_state_dict(ckpt["optimizer"])
        scheduler.load_state_dict(ckpt["scheduler"])
        start_epoch      = ckpt["epoch"]
        start_step       = ckpt.get("step", -1) + 1
        best_bleu        = ckpt.get("best_bleu", 0.0)
        log_history      = ckpt.get("log_history", [])
        epoch_loss_accum = ckpt.get("epoch_loss_accum", 0.0)
        print(f"  → Tiếp tục từ epoch {start_epoch + 1}, step {start_step}")
    else:
        print("  → Không tìm thấy checkpoint, train từ đầu")

    # ── Train ──────────────────────────────────────────────
    save_interval  = CFG["save_every_minutes"] * 60
    last_save_time = time.time()

    print(f"\n{'='*60}")
    print(f"Training đa ngôn ngữ ({len(CFG['lang_pairs'])*2} chiều dịch)")
    print(f"Epochs  : {CFG['epochs']} | Steps/epoch: {len(train_loader)}")
    print(f"Warmup  : {warmup_steps} → Total: {total_steps} steps")
    print(f"Save    : mỗi {CFG['save_every_minutes']} phút")
    print(f"{'='*60}\n")

    for epoch in range(start_epoch, CFG["epochs"]):
        model.train()
        optimizer.zero_grad()
        epoch_loss = epoch_loss_accum
        t0 = time.time()

        for step, batch in enumerate(train_loader):

            # Skip các step đã train khi resume
            if epoch == start_epoch and step < start_step:
                if step > 0 and step % 500 == 0:
                    print(f"  ⏩ Skip {step}/{start_step - 1}...")
                continue

            input_ids      = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            labels         = batch["labels"].to(device)

            with torch.cuda.amp.autocast(dtype=torch.bfloat16,
                                         enabled=device.type == "cuda"):
                out  = model(input_ids=input_ids,
                             attention_mask=attention_mask, labels=labels)
                loss = out.loss / CFG["grad_accum"]

            if torch.isnan(loss):
                print(f"  ⚠️  NaN tại E{epoch+1} step {step+1}, bỏ qua")
                optimizer.zero_grad()
                continue

            scaler.scale(loss).backward()
            epoch_loss += loss.item() * CFG["grad_accum"]

            if (step + 1) % CFG["grad_accum"] == 0:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(),
                                         CFG["max_grad_norm"])
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                optimizer.zero_grad()

                # Auto-save mỗi 10 phút
                if time.time() - last_save_time >= save_interval:
                    save_ckpt(
                        ckpt_path, epoch, step, epoch_loss,
                        model, optimizer, scheduler, best_bleu, log_history
                    )
                    elapsed_min = (time.time() - t0) / 60
                    print(f"  💾 Saved | E{epoch+1} step {step+1}/"
                          f"{len(train_loader)} | {elapsed_min:.1f} phút")
                    last_save_time = time.time()

            if (step + 1) % 100 == 0:
                steps_done = (step - start_step + 1
                              if epoch == start_epoch else step + 1)
                lr_now = scheduler.get_last_lr()[0]
                print(
                    f"  [E{epoch+1}] {step+1}/{len(train_loader)} "
                    f"| loss={epoch_loss/max(steps_done,1):.4f} "
                    f"| lr={lr_now:.2e} | {time.time()-t0:.0f}s"
                )

        # ── Validation ─────────────────────────────────────
        avg_train      = epoch_loss / len(train_loader)
        val_loss, bleu = validate(model, val_loader, dec_tokenizer, device)
        elapsed        = time.time() - t0

        print(f"\n{'─'*50}")
        print(f"[Epoch {epoch+1}/{CFG['epochs']}]")
        print(f"  Train Loss : {avg_train:.4f}")
        print(f"  Val Loss   : {val_loss:.4f}  (PPL={math.exp(val_loss):.2f})")
        print(f"  BLEU       : {bleu:.2f}  (avg tất cả chiều dịch)")
        print(f"  Thời gian  : {elapsed/60:.1f} phút")
        print(f"{'─'*50}\n")

        log_history.append({
            "epoch":      epoch + 1,
            "train_loss": round(avg_train, 4),
            "val_loss":   round(val_loss, 4),
            "ppl":        round(math.exp(val_loss), 2),
            "bleu":       round(bleu, 2),
            "time_min":   round(elapsed / 60, 1),
        })
        with open(CFG["log_file"], "w") as f:
            json.dump(log_history, f, indent=2, ensure_ascii=False)

        # Save cuối epoch
        save_ckpt(ckpt_path, epoch + 1, -1, 0.0,
                  model, optimizer, scheduler, best_bleu, log_history)
        last_save_time = time.time()
        start_step       = 0
        epoch_loss_accum = 0.0

        if bleu > best_bleu:
            best_bleu = bleu
            save_ckpt(
                os.path.join(CFG["output_dir"], "best_model.pt"),
                epoch + 1, -1, 0.0,
                model, optimizer, scheduler, best_bleu, log_history
            )
            print(f"  ✅ Best model saved! BLEU={bleu:.2f}")

    print(f"\n{'='*60}")
    print(f"Training hoàn tất! Best BLEU = {best_bleu:.2f}")
    print(f"Model lưu tại: {CFG['output_dir']}")
    print(f"{'='*60}")


if __name__ == "__main__":
    # from google.colab import drive; drive.mount("/content/drive")
    train()


## 📝 Cell 5 – Ghi test.py

In [ ]:
%%writefile /content/test.py
"""
test.py – Inference & Đánh giá đa ngôn ngữ (6 chiều dịch)
Hỗ trợ: en↔vi, en↔fr, vi↔fr

Cách dùng language token:
    ">>vi<< Hello"   → dịch sang tiếng Việt
    ">>en<< Bonjour" → dịch sang tiếng Anh
    ">>fr<< Xin chào"→ dịch sang tiếng Pháp
"""

import os, json, torch
from transformers import AutoTokenizer
from datasets import load_dataset
import sacrebleu

from model import TranslationModel

# ──────────────────────────────────────────────────────────────
# 1. CẤU HÌNH
# ──────────────────────────────────────────────────────────────
CFG = {
    "model_path":     "/content/drive/MyDrive/translation_model/best_model.pt",
    "encoder_name":   "jhu-clsp/mmBERT-small",
    "decoder_name":   "google/mt5-small",
    "max_new_tokens": 128,
    "num_beams":      4,
    "length_penalty": 1.0,
    "test_samples":   300,   # mỗi chiều dịch
    "batch_size":     32,
}

# 6 chiều dịch cần đánh giá
EVAL_PAIRS = [
    ("en", "vi"),
    ("vi", "en"),
    ("en", "fr"),
    ("fr", "en"),
    ("vi", "fr"),
    ("fr", "vi"),
]


# ──────────────────────────────────────────────────────────────
# 2. LOAD MODEL
# ──────────────────────────────────────────────────────────────
def load_model(cfg, device):
    print(f"Load model từ: {cfg['model_path']}")
    ckpt  = torch.load(cfg["model_path"], map_location=device)
    model = TranslationModel(
        encoder_name=cfg["encoder_name"],
        decoder_name=cfg["decoder_name"],
    ).to(device)
    model.load_state_dict(ckpt["model"])
    model.eval()
    print("  ✅ Load thành công!")
    if "log_history" in ckpt and ckpt["log_history"]:
        best = max(ckpt["log_history"], key=lambda x: x["bleu"])
        print(f"  → Best epoch {best['epoch']} | "
              f"BLEU={best['bleu']} | PPL={best.get('ppl','?')}")
    return model


# ──────────────────────────────────────────────────────────────
# 3. DỊCH MỘT CÂU
# ──────────────────────────────────────────────────────────────
def translate_sentence(text, src_lang, tgt_lang,
                       model, enc_tokenizer, dec_tokenizer, device,
                       num_beams=4, max_new_tokens=128, length_penalty=1.0):
    """
    Dịch một câu từ src_lang sang tgt_lang.
    Language token tự động được ghép vào đầu câu nguồn.
    """
    # Ghép language token: ">>tgt<< câu nguồn"
    src_with_token = TranslationModel.add_lang_token(text, tgt_lang)

    enc = enc_tokenizer(src_with_token, max_length=130,
                        padding="max_length", truncation=True,
                        return_tensors="pt")
    input_ids      = enc["input_ids"].to(device)
    attention_mask = enc["attention_mask"].to(device)

    with torch.no_grad():
        output_ids = model.translate(
            input_ids=input_ids, attention_mask=attention_mask,
            max_new_tokens=max_new_tokens,
            num_beams=num_beams, length_penalty=length_penalty,
        )
    return dec_tokenizer.decode(output_ids[0], skip_special_tokens=True)


# ──────────────────────────────────────────────────────────────
# 4. ĐÁNH GIÁ BLEU TỪNG CHIỀU DỊCH
# ──────────────────────────────────────────────────────────────
def evaluate_direction(src_lang, tgt_lang,
                       model, enc_tokenizer, dec_tokenizer, device, cfg):
    """Đánh giá BLEU cho một chiều dịch cụ thể."""
    key = f"{src_lang}-{tgt_lang}"
    rev = f"{tgt_lang}-{src_lang}"
    print(f"\n  [{src_lang}→{tgt_lang}] Tải test set...")

    # Thử load theo cả 2 thứ tự
    ds = None
    for pair_key in [key, rev]:
        try:
            ds = load_dataset("Helsinki-NLP/opus-100",
                              pair_key, split="test")
            break
        except Exception:
            continue

    if ds is None:
        print(f"  ⚠️  Không tìm thấy test set cho {key}, bỏ qua")
        return None, None, None

    if cfg["test_samples"]:
        ds = ds.select(range(min(cfg["test_samples"], len(ds))))

    sources, references = [], []
    for item in ds:
        tr = item["translation"]
        s  = tr.get(src_lang, "")
        t  = tr.get(tgt_lang, "")
        if s and t:
            sources.append(s)
            references.append(t)

    predictions = []
    BS = cfg["batch_size"]
    model.eval()

    for i in range(0, len(sources), BS):
        batch_src = sources[i : i + BS]

        # Ghép language token cho cả batch
        batch_with_token = [
            TranslationModel.add_lang_token(s, tgt_lang)
            for s in batch_src
        ]

        enc = enc_tokenizer(batch_with_token, max_length=130,
                            padding="max_length", truncation=True,
                            return_tensors="pt")
        input_ids      = enc["input_ids"].to(device)
        attention_mask = enc["attention_mask"].to(device)

        with torch.no_grad():
            output_ids = model.translate(
                input_ids=input_ids, attention_mask=attention_mask,
                max_new_tokens=cfg["max_new_tokens"],
                num_beams=cfg["num_beams"],
                length_penalty=cfg["length_penalty"],
            )
        predictions.extend(
            dec_tokenizer.batch_decode(output_ids, skip_special_tokens=True)
        )

    bleu = sacrebleu.corpus_bleu(predictions, [references]).score
    print(f"  [{src_lang}→{tgt_lang}] BLEU = {bleu:.2f}  ({len(sources)} câu)")
    return predictions, references, bleu


def evaluate_all_directions(model, enc_tokenizer, dec_tokenizer, device, cfg):
    """Đánh giá tất cả 6 chiều dịch và in bảng tổng hợp."""
    print("\n" + "="*60)
    print("ĐÁNH GIÁ BLEU – TẤT CẢ CÁC CHIỀU DỊCH")
    print("="*60)

    results = {}
    all_preds_refs = {}

    for src_lang, tgt_lang in EVAL_PAIRS:
        preds, refs, bleu = evaluate_direction(
            src_lang, tgt_lang,
            model, enc_tokenizer, dec_tokenizer, device, cfg
        )
        if bleu is not None:
            results[f"{src_lang}→{tgt_lang}"] = round(bleu, 2)
            all_preds_refs[f"{src_lang}→{tgt_lang}"] = {
                "predictions": preds[:20],
                "references":  refs[:20],
            }

    # Bảng tổng hợp
    print(f"\n{'='*40}")
    print(f"  {'Chiều dịch':15s} | {'BLEU':>6}")
    print(f"  {'-'*25}")
    for direction, bleu in results.items():
        print(f"  {direction:15s} | {bleu:>6.2f}")
    if results:
        avg = sum(results.values()) / len(results)
        print(f"  {'-'*25}")
        print(f"  {'Trung bình':15s} | {avg:>6.2f}")
    print(f"{'='*40}")

    return results, all_preds_refs


# ──────────────────────────────────────────────────────────────
# 5. DEMO NHANH – TẤT CẢ CHIỀU DỊCH
# ──────────────────────────────────────────────────────────────
def demo_all_directions(model, enc_tokenizer, dec_tokenizer, device):
    """Demo dịch một câu qua tất cả các chiều."""

    test_sentences = {
        "en": "Artificial intelligence is transforming the world.",
        "vi": "Trí tuệ nhân tạo đang thay đổi thế giới.",
        "fr": "L'intelligence artificielle transforme le monde.",
    }

    print("\n" + "="*60)
    print("DEMO – 6 CHIỀU DỊCH")
    print("="*60)

    for src_lang, tgt_lang in EVAL_PAIRS:
        src_text = test_sentences[src_lang]
        result   = translate_sentence(
            src_text, src_lang, tgt_lang,
            model, enc_tokenizer, dec_tokenizer, device,
        )
        lang_names = {"en": "English", "vi": "Tiếng Việt", "fr": "Français"}
        print(f"\n  {lang_names[src_lang]} → {lang_names[tgt_lang]}:")
        print(f"    IN : {src_text}")
        print(f"    OUT: {result}")


# ──────────────────────────────────────────────────────────────
# 6. INTERACTIVE MODE
# ──────────────────────────────────────────────────────────────
def interactive_translate(model, enc_tokenizer, dec_tokenizer, device):
    LANG_NAMES = {"en": "English", "vi": "Tiếng Việt", "fr": "Français"}
    LANGS      = list(LANG_NAMES.keys())

    print("\n" + "="*60)
    print("DỊCH TƯƠNG TÁC ĐA NGÔN NGỮ")
    print("="*60)
    print("Gõ 'quit' để thoát\n")

    while True:
        # Chọn ngôn ngữ nguồn
        print("Ngôn ngữ nguồn:", " | ".join(
            f"{i+1}.{l} ({LANG_NAMES[l]})" for i, l in enumerate(LANGS)
        ))
        src_choice = input("Chọn (1/2/3): ").strip()
        if src_choice.lower() == "quit": break
        try:
            src_lang = LANGS[int(src_choice) - 1]
        except Exception:
            print("Lựa chọn không hợp lệ"); continue

        # Chọn ngôn ngữ đích
        tgt_options = [l for l in LANGS if l != src_lang]
        print("Ngôn ngữ đích:", " | ".join(
            f"{i+1}.{l} ({LANG_NAMES[l]})" for i, l in enumerate(tgt_options)
        ))
        tgt_choice = input("Chọn (1/2): ").strip()
        if tgt_choice.lower() == "quit": break
        try:
            tgt_lang = tgt_options[int(tgt_choice) - 1]
        except Exception:
            print("Lựa chọn không hợp lệ"); continue

        # Nhập câu
        text = input(f"\n[{LANG_NAMES[src_lang]}] > ").strip()
        if not text: continue
        if text.lower() == "quit": break

        result = translate_sentence(
            text, src_lang, tgt_lang,
            model, enc_tokenizer, dec_tokenizer, device,
        )
        print(f"[{LANG_NAMES[tgt_lang]}] > {result}\n")


# ──────────────────────────────────────────────────────────────
# 7. MAIN
# ──────────────────────────────────────────────────────────────
if __name__ == "__main__":
    # from google.colab import drive; drive.mount("/content/drive")

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Device: {device}")

    enc_tokenizer = AutoTokenizer.from_pretrained(CFG["encoder_name"])
    dec_tokenizer = AutoTokenizer.from_pretrained(CFG["decoder_name"])
    model         = load_model(CFG, device)

    # ── A. Demo 6 chiều dịch ─────────────────────────────
    demo_all_directions(model, enc_tokenizer, dec_tokenizer, device)

    # ── B. Đánh giá BLEU tất cả chiều ────────────────────
    results, samples = evaluate_all_directions(
        model, enc_tokenizer, dec_tokenizer, device, CFG
    )

    # Lưu kết quả
    out_path = os.path.join(
        os.path.dirname(CFG["model_path"]), "test_results.json"
    )
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump({
            "bleu_per_direction": results,
            "avg_bleu": round(sum(results.values())/len(results), 2) if results else 0,
            "samples":  samples,
        }, f, ensure_ascii=False, indent=2)
    print(f"\nKết quả lưu: {out_path}")

    # ── C. Interactive (bỏ comment để dùng) ──────────────
    # interactive_translate(model, enc_tokenizer, dec_tokenizer, device)


## 🔍 Cell 6 – Kiểm tra checkpoint

In [ ]:
import torch, json, os

CKPT = "/content/drive/MyDrive/translation_model/checkpoint_latest.pt"

if os.path.exists(CKPT):
    ckpt = torch.load(CKPT, map_location="cpu")
    epoch     = ckpt.get("epoch", 0)
    step      = ckpt.get("step", -1)
    best_bleu = ckpt.get("best_bleu", 0.0)
    log       = ckpt.get("log_history", [])
    cfg       = ckpt.get("cfg", {})

    print(f"📋 Checkpoint info:")
    print(f"   Epoch đã lưu   : {epoch}")
    print(f"   Step đã lưu    : {step}")
    print(f"   Step tiếp theo : {step + 1}  (epoch {epoch + 1})")
    print(f"   Best BLEU      : {best_bleu:.2f}")
    print(f"   Lang pairs     : {cfg.get('lang_pairs', 'N/A')}")

    if log:
        print(f"\n📊 Lịch sử:")
        print(f"  {'Epoch':>5} | {'TrainLoss':>9} | {'ValLoss':>7} | {'PPL':>7} | {'BLEU':>6} | {'Phút':>5}")
        print("  " + "-"*50)
        for e in log:
            print(f"  {e['epoch']:>5} | {e['train_loss']:>9.4f} | "
                  f"{e['val_loss']:>7.4f} | {e.get('ppl',0):>7.2f} | "
                  f"{e['bleu']:>6.2f} | {e['time_min']:>5.1f}")
else:
    print("ℹ️  Không có checkpoint → train từ đầu")

## 🚀 Cell 7 – Train / Resume

> Auto-resume từ checkpoint. Auto-save mỗi **10 phút** + cuối epoch.
>
> Để thêm/bớt cặp ngôn ngữ, sửa `lang_pairs` trong `train.py` rồi chạy lại Cell 4.

In [ ]:
%cd /content
!python train.py

## 📊 Cell 8 – Đánh giá BLEU tất cả 6 chiều

In [ ]:
%cd /content
!python test.py

## 🧪 Cell 9 – Test nhanh một câu bất kỳ

In [ ]:
import torch, sys
sys.path.insert(0, "/content")
from transformers import AutoTokenizer
from model import TranslationModel

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

enc_tok = AutoTokenizer.from_pretrained("jhu-clsp/mmBERT-small")
dec_tok = AutoTokenizer.from_pretrained("google/mt5-small")

ckpt  = torch.load("/content/drive/MyDrive/translation_model/best_model.pt", map_location=device)
model = TranslationModel().to(device)
model.load_state_dict(ckpt["model"])
model.eval()

def quick_translate(text, src_lang, tgt_lang):
    src = TranslationModel.add_lang_token(text, tgt_lang)
    enc = enc_tok(src, return_tensors="pt",
                  max_length=130, padding="max_length", truncation=True)
    with torch.no_grad():
        out = model.translate(enc["input_ids"].to(device),
                              enc["attention_mask"].to(device),
                              num_beams=4)
    return dec_tok.decode(out[0], skip_special_tokens=True)

# ── Thay đổi câu và chiều dịch tại đây ──────────────────────
tests = [
    ("Hello, how are you?",               "en", "vi"),
    ("Trí tuệ nhân tạo rất thú vị.",       "vi", "en"),
    ("Bonjour, comment allez-vous ?",       "fr", "en"),
    ("Tôi yêu học ngôn ngữ mới.",          "vi", "fr"),
    ("Artificial intelligence is amazing.", "en", "fr"),
    ("La France est un beau pays.",         "fr", "vi"),
]

print(f"{'Chiều':8s} | {'Input':40s} | Output")
print("-"*90)
for text, src, tgt in tests:
    result = quick_translate(text, src, tgt)
    print(f"{src}→{tgt}  | {text[:40]:40s} | {result}")

## 📈 Cell 10 – Xem log training

In [ ]:
import json, os

LOG = "/content/drive/MyDrive/translation_model/train_log.json"
if os.path.exists(LOG):
    with open(LOG) as f: log = json.load(f)
    print(f"  {'Epoch':>5} | {'TrainLoss':>9} | {'ValLoss':>7} | {'PPL':>7} | {'BLEU':>6} | {'Phút':>5}")
    print("  " + "-"*50)
    for e in log:
        print(f"  {e['epoch']:>5} | {e['train_loss']:>9.4f} | "
              f"{e['val_loss']:>7.4f} | {e.get('ppl',0):>7.2f} | "
              f"{e['bleu']:>6.2f} | {e['time_min']:>5.1f}")
else:
    print("Chưa có log")

## ⚠️ Cell 11 – Xoá checkpoint (train lại từ đầu)
> Chỉ chạy khi muốn **reset hoàn toàn**!

In [ ]:
import os
for f in [
    "/content/drive/MyDrive/translation_model/checkpoint_latest.pt",
    "/content/drive/MyDrive/translation_model/best_model.pt",
]:
    if os.path.exists(f):
        os.remove(f); print(f"🗑️  Đã xoá: {f}")
    else:
        print(f"Không tìm thấy: {f}")
print("✅ Sẵn sàng train lại từ đầu")